# 01 — Análise Exploratória de Dados (EDA) — open-r1/codeforces

Este notebook gera:
1. Relatório passo a passo de filtragem, deduplicação de `aliases` (incluindo conflitos de rating/tags) e confirmação cronológica do conjunto de teste.
2. Tabela completa de frequência das tags no treino (antes de aplicar `min_tag_freq=50`).
3. Distribuição das 5 faixas de dificuldade (treino/val/teste), co-ocorrência e tags por problema.
4. Estatísticas de comprimento em tokens **para cada tokenizador** (média, mediana, p90, p95, p99, max, `%>512`, `%>2048` e `p95` do `input_format` com proposta de ajuste).
5. Baselines triviais (classe majoritária) para dificuldade e tags.
6. Tabelas `.csv`/`.tex` em `results/tables/` e gráficos `.pdf`/`.png` em `results/figures/`.


In [ ]:
import json
from pathlib import Path
import pandas as pd

from tcc.config import load_config
from tcc.data import load_or_prepare_splits
from tcc.eda import (
    analyze_token_lengths_by_tokenizer,
    compute_difficulty_distribution,
    compute_tag_cooccurrence,
    compute_tags_per_problem_stats,
    compute_trivial_baselines,
    plot_difficulty_distribution,
    plot_token_length_histograms,
    propose_input_format_budget,
    save_table_csv_and_tex,
)

cfg = load_config()
bundle = load_or_prepare_splits(cfg, force_rebuild=False)

train_df = bundle["train"]
val_df = bundle["val"]
test_df = bundle["test"]
tag_vocab = bundle["tag_vocab"]
train_tag_freq = bundle["train_tag_freq"]

print("=== 1. Contagem por etapa de filtro ===")
filter_df = pd.DataFrame(bundle["filter_steps"])
display(filter_df)

print("\n=== 2. Relatório de grupos de aliases e conflitos ===")
print(json.dumps(bundle["alias_report"], indent=2, ensure_ascii=False))

print("\n=== 3. Confirmação cronológica (contest_start_year) ===")
print(json.dumps(bundle["year_summary"], indent=2, ensure_ascii=False))

In [ ]:
# === 4. Tabela completa de frequência das tags no treino (antes de fixar min_tag_freq=50) ===
print(f"Total de tags distintas no treino (sem '*'): {len(train_tag_freq)}")
print(f"Tags selecionadas com min_tag_freq={cfg['labels']['min_tag_freq']}: {len(tag_vocab)}")
display(train_tag_freq)

save_table_csv_and_tex(train_tag_freq, cfg["paths"]["tables_dir"], "eda_tag_frequencies")

# Estatísticas de número de tags por problema
for split_name, frame in [("Treino", train_df), ("Validação", val_df), ("Teste", test_df)]:
    stats = compute_tags_per_problem_stats(frame, tag_vocab, cfg["labels"]["exclude_tag_prefix"])
    print(f"Tags/problema ({split_name}):", stats)

In [ ]:
# === 5. Distribuição das 5 faixas de dificuldade ===
dist_df = compute_difficulty_distribution(
    train_df, val_df, test_df, cfg["labels"]["difficulty_names"]
)
display(dist_df)

save_table_csv_and_tex(dist_df, cfg["paths"]["tables_dir"], "eda_difficulty_distribution")
plot_difficulty_distribution(dist_df, cfg["paths"]["figures_dir"])

# Co-ocorrência das top-15 tags no treino
top15_tags = train_tag_freq.head(15)["tag"].tolist()
cooc_df = compute_tag_cooccurrence(train_df, top15_tags, cfg["labels"]["exclude_tag_prefix"])
display(cooc_df)
cooc_df.to_csv(Path(cfg["paths"]["tables_dir"]) / "eda_tag_cooccurrence_top15.csv")

In [ ]:
# === 6. Comprimento em tokens POR TOKENIZADOR (geral, por faixa e p95 do input_format) ===
all_df = pd.concat([train_df, val_df, test_df], ignore_index=True)
overall_len_df, by_class_len_df, raw_lengths = analyze_token_lengths_by_tokenizer(
    all_df,
    model_specs=cfg["models"],
    fields=cfg["tokenization"]["fields"],
    difficulty_names=cfg["labels"]["difficulty_names"],
    cache_dir=cfg["paths"]["cache_dir"],
)

print("=== Comprimento geral por tokenizador ===")
display(overall_len_df)
print("\nRecomendação para reserve_input_format_tokens:")
print(propose_input_format_budget(overall_len_df, cfg["tokenization"]["reserve_input_format_tokens"]))

print("\n=== Comprimento por faixa de dificuldade ===")
display(by_class_len_df)

save_table_csv_and_tex(overall_len_df, cfg["paths"]["tables_dir"], "eda_token_lengths_overall")
save_table_csv_and_tex(by_class_len_df, cfg["paths"]["tables_dir"], "eda_token_lengths_by_class")
plot_token_length_histograms(raw_lengths, cfg["paths"]["figures_dir"])

In [ ]:
# === 7. Baseline trivial (classe majoritária) ===
baselines = compute_trivial_baselines(
    train_df, test_df, tag_vocab, cfg["labels"]["exclude_tag_prefix"]
)
print(json.dumps(baselines, indent=2, ensure_ascii=False))

baselines_path = Path(cfg["paths"]["tables_dir"]) / "eda_trivial_baselines.json"
baselines_path.write_text(json.dumps(baselines, indent=2, ensure_ascii=False), encoding="utf-8")
print("Salvo em:", baselines_path)